# Logistic Regression: локальное обучение и проверка

Этот notebook запускает тот же src.run и те же конфигурации 001–006, что и 00_kaggle.ipynb. 001–004 сравнивают preprocessing и веса, 005 выбирает основу по tuned OOF, 006 добавляет 60/90 s к 005. Каждый подход: C=1, затем поиск C на тех же трёх folds, затем refit на train и оценка val. Выбор основы и C не использует val.

Перед первым запуском выполните uv sync --locked в корне проекта. Затем выполните Run All. Данные автоматически ищутся в data/raw/ и data/; при другом расположении задайте DATA_ROOT явно.

**SMOKE=False** запускает полную серию. Для предварительной проверки установите True: девять людей train, два val, короткие участки вокруг стадий; это проверка работоспособности, не качества полного эксперимента. Обучение вызывается через uv в закреплённом окружении, независимо от Python текущего ядра. Конфигурации запускаются последовательно.

In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
if not (DATA_ROOT / "labels.csv").is_file():
    DATA_ROOT = PROJECT_ROOT / "data"
for name in ("labels.csv", "epochs/train", "epochs/val"):
    if not (DATA_ROOT / name).exists():
        raise FileNotFoundError(f"Укажите DATA_ROOT: отсутствует {DATA_ROOT / name}")
OUTPUT_ROOT = PROJECT_ROOT / "artefacts"  # Путь должен быть доступен для записи.
SMOKE = False
CONFIGS = [PROJECT_ROOT / "configs" / f"{i:03d}_linear.yaml" for i in range(1, 7)]
command = [
    "uv",
    "run",
    "--locked",
    "python",
    "-m",
    "src.run",
    *map(str, CONFIGS),
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
]
if SMOKE:
    command.append("--smoke")
print("Данные:", DATA_ROOT)
print("Результаты:", OUTPUT_ROOT, "SMOKE:", SMOKE)

Первый запуск извлекает признаки. Кэш 30 s общий для всех подходов; длинные окна нужны только 006. Подбор C не пересчитывает признаки. Кэш учитывает код извлечения, параметры, версии библиотек, метки, размеры и время изменения файлов. MANIFEST.json не требуется; сверка с исходными контрольными суммами не выполняется. Проверки ID, групп людей, каналов, формы и конечности загружаемых сигналов сохраняются.

Повторный запуск использует завершённые trials при идентичных настройках, данных и fingerprint кода; завершённый эксперимент пропускается. Незавершённый trial C (все его folds) и незавершённый refit обучаются заново. Обучаемый preprocessing заново выполняется внутри каждого fold; общего scaler для train и val нет. Изменённый эксперимент не перезаписывается.

Границы найденного C допускают одну дополнительную декаду; все trials сохраняются. OOF после подбора оптимистичен и не является независимой оценкой. Не меняйте выбор конфигурации по val в ходе этого запуска.

In [ ]:
from src.notebook import run_command

run_command(command, cwd=PROJECT_ROOT)

Результаты — в OUTPUT_ROOT/<ID>/, smoke — в OUTPUT_ROOT/smoke/<ID>/. Следующая ячейка показывает fixed/tuned, выбранный C, OOF и выданную val для всех шести подходов. Метрики классов и confusion matrix сохраняются в metrics.json; подробное сравнение находится в 03_eval. В experiments.csv записываются только полноценные завершённые запуски. При смене кода/зависимостей используйте отдельный output root для новых smoke-проверок.

In [ ]:
namespace = OUTPUT_ROOT / "smoke" if SMOKE else OUTPUT_ROOT
summary_code = """
import sys
from pathlib import Path
import polars as pl
from src.evaluation import comparison_table

summary = comparison_table(Path(sys.argv[1]), sys.argv[2:])
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(summary.select("experiment_id", "regime", "C", "oof_macro_f1", "val_macro_f1"))
"""
run_command(
    [
        "uv",
        "run",
        "--locked",
        "python",
        "-c",
        summary_code,
        str(namespace),
        *[path.stem for path in CONFIGS],
    ],
    cwd=PROJECT_ROOT,
)